In [0]:
# 1024/128

In [0]:
spark.conf.set("spark.databricks.optimizer.adaptive.enabled", "false")
spark.conf.set("spark.databricks.adaptive.autoBroadcastJoinThreshold", "-1")

In [0]:
from pyspark.sql import functions as F

In [0]:
orders_df = (
    spark.range(0, 100_00_000)
    .withColumn(
        "country",
        F.when(F.col("id") < 90_00_000, "India")
        .when(F.col("id") < 90_50_000, "USA")
        .when(F.col("id") < 90_70_000, "Canada")
        .otherwise("UK")
    )
    .withColumn("amount", (F.rand() * 1000).cast("int"))
    .withColumn("date", F.current_date())
)

display(orders_df)

In [0]:
orders_df.groupBy("country").count().show()

In [0]:
orders_df = orders_df.repartition(4, "country")

In [0]:
cus_df = (
    spark.createDataFrame(
        [
            ("India", "Asia"),
            ("USA", "South America"),
            ("Canada", "North America"),
            ("UK", "Europe"),
        ]
        , ["country", "continent"]
    )
)

display(cus_df)

In [0]:
# perform a join

result_df = (
    orders_df
    .join(
        cus_df, 
        on=["country"],
        how="inner"
    )
)
# action to trigger
result_df.show()

In [0]:
# perform a broadcast join

result_df = (
    orders_df
    .join(
        F.broadcast(cus_df), 
        on=["country"],
        how="inner"
    )
)
# action to trigger
result_df.show()

In [0]:
# 2. Salting Technique

orders_df = orders_df.withColumn("salt", (F.rand()*10).cast("int"))
cus_df = cus_df.withColumn("salt", F.explode(F.array(*[F.lit(i) for i in range(10)])))

In [0]:
cus_df.display()

In [0]:
# perform a join using salting technique
orders_df = orders_df.repartition(50, "country", "salt")
result_df = (
    orders_df
    .join(
        cus_df, 
        on=["country"],
        how="inner"
    )
)
# action to trigger
result_df.show()